# Scalar autograd: PyTorch, from scratch, then a fused sigmoid

We will start with one graph:

$$
m=wx,\qquad a=m+b,\qquad e=a-y,\qquad L=e^2
$$

with $w=2$, $x=3$, $b=1$, and $y=10$.

First we let PyTorch differentiate it. Then we build the smallest useful version of the same idea
ourselves. The point is not to replace PyTorch—it is to see what `.backward()` does. A final optional
example then compares one fused sigmoid operation with the same sigmoid expanded into atomic operations.

<div style="max-width:100%;overflow-x:auto;margin:1.25rem 0 0.5rem;padding:0.25rem 0;">
  
</div>
<p style="margin:0.25rem 0 1.25rem;color:#52696D;font-size:0.95rem;">
  Forward computes left → right. Backward begins at <code>L.grad = 1</code> and travels right → left.
  The teal number in each box is that node's final <code>∂L/∂node</code>. On a phone, scroll sideways.
</p>

## 1 · The whole example in PyTorch

Each number below is a scalar tensor. We set `requires_grad=True` because we want PyTorch to calculate
its loss derivative. In ordinary training, the target `y` would normally be fixed; here we track it only
so the output matches our complete paper calculation.

PyTorch keeps `.grad` automatically for the four leaf tensors. `retain_grad()` asks it to keep gradients
for the intermediate values too.

In [1]:
import torch

torch.set_default_dtype(torch.float64)

w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(10.0, requires_grad=True)

# Forward pass
m = w * x
a = m + b
e = a - y
L = e ** 2

for node in (m, a, e, L):
    node.retain_grad()

print("Forward: m =", m.item(), ", a =", a.item(),
      ", e =", e.item(), ", L =", L.item())

Forward: m = 6.0 , a = 7.0 , e = -3.0 , L = 9.0


Now the important line:

In [2]:
L.backward()

print("w.grad =", w.grad)
print("x.grad =", x.grad)
print("b.grad =", b.grad)
print("y.grad =", y.grad)

w.grad = tensor(-18.)
x.grad = tensor(-12.)
b.grad = tensor(-6.)
y.grad = tensor(6.)


That is PyTorch autograd. The forward pass created a graph; `.backward()` sent a seed gradient of $1$
from $L$ through that graph in reverse.

For comparison with the paper calculation, here is every stored value:

In [3]:
torch_nodes = {"w": w, "x": x, "m": m, "b": b,
               "a": a, "y": y, "e": e, "L": L}
torch_reference = {
    name: (node.item(), node.grad.item())
    for name, node in torch_nodes.items()
}

print(f"{'node':<5} {'value':>8} {'grad':>8}")
for name, (value, grad) in torch_reference.items():
    print(f"{name:<5} {value:8.1f} {grad:8.1f}")

expected = {
    "w": (2, -18), "x": (3, -12), "m": (6, -6), "b": (1, -6),
    "a": (7, -6), "y": (10, 6), "e": (-3, -6), "L": (9, 1),
}
assert torch_reference == expected

node     value     grad
w          2.0    -18.0
x          3.0    -12.0
m          6.0     -6.0
b          1.0     -6.0
a          7.0     -6.0
y         10.0      6.0
e         -3.0     -6.0
L          9.0      1.0


## 2 · A tiny autograd engine from scratch

Suppose one operation $f$ takes a value $u$ and produces $v=f(u)$. In this notebook's convention,
the **forward construction** makes $u$ a direct **parent** (or operand) of $v$, while $v$ is the output
(or child) created from $u$. These words describe one operation in the computation graph—not a whole
neural-network layer. Libraries sometimes choose different names; here, follow `v.parents`, whose links
point back to the direct operands.

<div style="max-width:100%;overflow-x:auto;margin:1.25rem 0 0.5rem;padding:0.25rem 0;">
  
</div>
<p style="margin:0.25rem 0 0.75rem;color:#52696D;font-size:0.95rem;">
  <code>v.parents</code> owns the <code>ParentLink</code>, and that link points back to <code>u</code>.
  The engine does not need <code>u</code> to keep a list of its children. On a phone, scroll sideways.
</p>
<div style="margin:0.4rem 0 1.25rem;padding:0.85rem 1rem;border-left:4px solid #2C7A7B;background:#F4FAF9;border-radius:0 10px 10px 0;color:#29464B;">
  <strong>Concrete link from our graph:</strong> in <code>m = w × x</code>, choose <code>v = m</code> and the parent <code>u = w</code>.
  Backward reads upstream <code>m.grad = −6</code>, reads local <code>∂m/∂w = x = 3</code> from that link,
  computes the temporary contribution <code>−6 × 3 = −18</code>, and adds it to <code>w.grad</code>.
  The other parent link uses <code>u = x</code>, local derivative <code>w = 2</code>, and contributes <code>−12</code> to <code>x.grad</code>.
</div>

For the autograd calculation, a `Value` needs only:

- its number in `data`,
- its accumulated loss-gradient buffer in `grad`, and
- ordered links to the operands that directly produced it.

Our teaching class also stores `label` and `op` so diagrams can say “m” and “×”. They are display
metadata: changing those strings does not change the forward number or any gradient.

Each `ParentLink` has exactly two fields: `.value` points to the parent operand, and `.local_grad` holds
the evaluated local derivative along that edge. For example, $m=wx$ remembers
$(w,\partial m/\partial w=x)$ and $(x,\partial m/\partial x=w)$.

The gradient names are always relative to the operation currently running:

- <span style="color:#2C7A7B;font-weight:700">upstream</span>:
  $g_v=\partial L/\partial v$, already accumulated in `v.grad`;
- <span style="color:#2B6CB0;font-weight:700">local</span>:
  $\partial v/\partial u$, stored in the link from output $v$ to parent $u$;
- <span style="color:#EB811B;font-weight:700">edge contribution to the parent</span>
  (the “downstream contribution” in our color legend):
  $\Delta g_u=g_v(\partial v/\partial u)$, computed during backward and added to `u.grad`.

The edge contribution is temporary: only the accumulated result in `u.grad` remains. `w` is a leaf
because it has no dependencies. `L` is the forward output or sink; `backward(L)` treats it as the
starting node—the root of the reverse traversal.

### Before writing `backward`: decide when a node is ready

A node must not send its gradient to its parents until **all gradient contributions arriving at that
node have been added to its `.grad` buffer**. We therefore need a dependency-safe processing order.
Start at `L`, follow its saved `ParentLink`s toward the inputs, and append each node only after all its
parents have been appended. Reversing the resulting list gives the safe order for backward.

<div style="max-width:100%;overflow-x:auto;margin:1.25rem 0 0.5rem;padding:0.25rem 0;">
  
</div>
<p style="margin:0.25rem 0 1.25rem;color:#52696D;font-size:0.95rem;">
  The formal name for any ordering that puts every dependency before the value that uses it is a
  <strong>topological order</strong>. We first understand the readiness rule; the name is secondary.
  On a phone, scroll sideways.
</p>

First apply the append-after-parents rule to one small part of the graph:

```text
visit(m):
  visit(w) → w has no parents → append w
  visit(x) → x has no parents → append x
  both parents are ready       → append m
```

Starting from `L` applies that same rule recursively to the whole graph. With our stored parent order,
the traversal produces exactly

```text
dependency-first:  w, x, m, b, a, y, e, L
process back:  L, e, y, a, b, m, x, w
```

Other dependency-safe lists are possible—for example, two independent leaves can swap places.
`seen` matters when a value feeds several later operations: following links from `L` may reach that same
object more than once, but it must be appended and processed only once. `seen` deduplicates `Value`
objects in the node schedule—not edges. If an operation is `w * w`, it still stores two `ParentLink`s,
and backward still processes both contributions. For $r=w^2$ at $w=2$, `w` appears once in the node
schedule, but the two links each contribute $2$; `w.grad += contribution` therefore gives $4$.

In [4]:
class ParentLink:
    def __init__(self, value, local_grad):
        self.value = value
        self.local_grad = float(local_grad)

class Value:
    def __init__(self, data, label="", parents=(), op=""):
        self.data = float(data)
        self.grad = 0.0
        self.label = label
        self.parents = tuple(parents)
        self.op = op

def multiply(u, v, label):
    return Value(u.data * v.data, label,
                 parents=(ParentLink(u, v.data),
                          ParentLink(v, u.data)), op="×")

def add(u, v, label):
    return Value(u.data + v.data, label,
                 parents=(ParentLink(u, 1.0),
                          ParentLink(v, 1.0)), op="+")

def subtract(u, v, label):
    return Value(u.data - v.data, label,
                 parents=(ParentLink(u, 1.0),
                          ParentLink(v, -1.0)), op="−")

def square(u, label):
    return Value(u.data ** 2, label,
                 parents=(ParentLink(u, 2 * u.data),), op="²")

def dependency_safe_order(root):
    """Return reachable Values with every parent before its output."""
    safe_order = []
    seen = set()

    def append_after_parents(node):
        # A shared Value may be reachable from the loss by several paths.
        # Visit and append that object only once.
        if id(node) in seen:
            return
        seen.add(id(node))

        # Follow output -> ParentLink -> operand, starting from the loss.
        for link in node.parents:
            append_after_parents(link.value)

        # Only now are all direct parents earlier in safe_order.
        safe_order.append(node)

    append_after_parents(root)
    return safe_order

def backward(root):
    safe_order = dependency_safe_order(root)

    # 1. Clear old accumulated gradients, then seed the loss.
    for node in safe_order:
        node.grad = 0.0
    root.grad = 1.0

    # 2. Reverse the safe order. A node's full upstream gradient is
    # ready before that node sends contributions to its parents.
    steps = []
    for output in reversed(safe_order):
        # One saved parent link gives one chain-rule update.
        for link in output.parents:
            parent = link.value
            upstream = output.grad
            local = link.local_grad
            contribution = upstream * local
            before = parent.grad
            parent.grad += contribution

            # Keep a teaching trace; autograd only needs the update above.
            steps.append({
                "output": output.label,
                "upstream": upstream,
                "parent": parent.label,
                "local": local,
                "downstream": contribution,
                "before": before,
                "after": parent.grad,
            })
    return steps

The visible code separates **finding a safe order** from **doing the calculus**:

1. `dependency_safe_order(root)` starts at `L`. `append_after_parents` follows each stored link to a
   direct operand and calls itself there first. Only after those calls return does it append the current
   node. `seen` makes a shared object a no-op on its second visit.
2. `backward(root)` clears every reachable `.grad`, then seeds `L.grad = 1` because
   $\partial L/\partial L=1$.
3. `reversed(safe_order)` processes `L, e, y, a, b, m, x, w`. At every saved link from output $v$
   to parent $u$, it reads the now-complete upstream gradient from `v.grad`, multiplies by the saved
   local derivative, and accumulates the result in `u.grad`.

Leaves such as `w` still appear in the processing list. They simply have no parent links, so there is
nothing further to update when their turn arrives.

| quantity | where it lives |
|---|---|
| upstream $g_v$ | already accumulated in `v.grad` |
| local $\partial v/\partial u$ | saved in `link.local_grad` during the forward pass |
| edge contribution to parent $\Delta g_u$ | temporary variable `contribution` for this one edge |
| accumulated $g_u$ | updated in `parent.grad` |

The autograd graph does **not** store a separate downstream gradient forever. It computes one edge
contribution, adds it to the parent's buffer, and that buffer later becomes the upstream gradient for
the parent. Our returned `steps` list is only a teaching log: it copies each contribution and the
before/after values so we can display them.

Three deliberate boundaries keep this engine small:

- it assumes an acyclic computation graph (a DAG);
- it seeds a scalar loss with `1`; vector outputs would need an explicit upstream seed;
- it clears reachable `.grad` buffers at the start of every call. PyTorch normally **accumulates**
  gradients across `.backward()` calls until you clear them.

In [5]:
#| echo: false
import importlib.util
import json
import subprocess
import sys
import warnings
from html import escape as escape_html

TOPOLOGY_SORT_ANIMATION_DOCUMENT = '<!doctype html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width,initial-scale=1">\n  <title>Dependency-safe ordering trace</title>\n</head>\n<body style="margin:0;padding:8px;background:#FFFFFF;">\n<section data-toposort-animation aria-label="Interactive trace of dependency-safe ordering"\n         tabindex="0" style="margin:0;">\n  <style>\n    [data-toposort-animation] {\n      --tsa-ink:#1F3A40; --tsa-muted:#52696D; --tsa-line:#C9D5D7;\n      --tsa-paper:#FFFFFF; --tsa-soft:#F4F7F7; --tsa-teal:#2C7A7B;\n      --tsa-teal-soft:#E8F7F5; --tsa-blue:#2B6CB0; --tsa-blue-soft:#EAF2FC;\n      --tsa-orange:#B35F0B;\n      --tsa-orange-soft:#FFF1E5; color:var(--tsa-ink);\n      font-family:Inter,ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;\n    }\n    [data-toposort-animation] * { box-sizing:border-box; }\n    [data-toposort-animation] .tsa-shell {\n      border:1px solid #D8E1E2; border-radius:18px; background:var(--tsa-paper);\n      box-shadow:0 10px 30px rgba(31,58,64,.07); overflow:hidden;\n    }\n    [data-toposort-animation] .tsa-heading { padding:20px 22px 16px; border-bottom:1px solid #E2E9EA; }\n    [data-toposort-animation] .tsa-kicker {\n      margin:0 0 5px; color:var(--tsa-blue); font-size:.76rem; font-weight:800;\n      letter-spacing:.08em; text-transform:uppercase;\n    }\n    [data-toposort-animation] h4 { margin:0; color:var(--tsa-ink); font-size:1.2rem; line-height:1.3; }\n    [data-toposort-animation] .tsa-intro { margin:7px 0 0; color:var(--tsa-muted); line-height:1.55; max-width:78ch; }\n    [data-toposort-animation] .tsa-controls {\n      display:flex; flex-wrap:wrap; align-items:center; gap:8px; padding:12px 14px;\n      background:#F8FAFA; border-bottom:1px solid #E2E9EA;\n    }\n    [data-toposort-animation] button, [data-toposort-animation] select {\n      min-height:44px; border:1px solid #71878A; border-radius:9px; background:#FFFFFF;\n      color:var(--tsa-ink); font:inherit; font-size:.9rem; font-weight:700;\n    }\n    [data-toposort-animation] button { padding:7px 12px; cursor:pointer; }\n    [data-toposort-animation] button[data-action="play"] {\n      min-width:104px; color:#FFFFFF; background:var(--tsa-teal); border-color:var(--tsa-teal);\n    }\n    [data-toposort-animation] button:hover:not(:disabled) { filter:brightness(.96); }\n    [data-toposort-animation] button:active:not(:disabled) { transform:translateY(1px); }\n    [data-toposort-animation] button:focus-visible,\n    [data-toposort-animation] select:focus-visible,\n    [data-toposort-animation]:focus-visible { outline:3px solid rgba(43,108,176,.3); outline-offset:2px; }\n    [data-toposort-animation] button:disabled { cursor:not-allowed; opacity:.42; }\n    [data-toposort-animation] .tsa-speed { display:flex; align-items:center; gap:6px; margin-left:4px; color:var(--tsa-muted); font-size:.86rem; font-weight:700; }\n    [data-toposort-animation] select { padding:6px 28px 6px 9px; }\n    [data-toposort-animation] .tsa-step-count { margin-left:auto; color:var(--tsa-muted); font-size:.84rem; font-variant-numeric:tabular-nums; }\n    [data-toposort-animation] .tsa-main {\n      display:grid; grid-template-columns:minmax(0,1.62fr) minmax(270px,.78fr); min-width:0;\n    }\n    [data-toposort-animation] .tsa-visual { min-width:0; padding:16px; border-right:1px solid #E2E9EA; }\n    [data-toposort-animation] .tsa-action {\n      display:grid; grid-template-columns:auto minmax(0,1fr); align-items:start; gap:10px;\n      min-height:70px; padding:12px 13px; border:1px solid #D8E1E2; border-radius:12px; background:#FBFCFC;\n    }\n    [data-toposort-animation] .tsa-action-badge,\n    [data-toposort-animation] .tsa-legend-badge {\n      display:inline-flex; align-items:center; justify-content:center; min-width:74px; padding:5px 8px;\n      border-radius:999px; font-size:.7rem; line-height:1; font-weight:850; letter-spacing:.045em; text-transform:uppercase;\n    }\n    [data-toposort-animation] .tsa-action-badge[data-kind="ready"] { color:#52696D; background:#E9EEEE; }\n    [data-toposort-animation] .tsa-action-badge[data-kind="enter"] { color:#235893; background:var(--tsa-blue-soft); }\n    [data-toposort-animation] .tsa-action-badge[data-kind="follow"] { color:#8B4A08; background:var(--tsa-orange-soft); }\n    [data-toposort-animation] .tsa-action-badge[data-kind="skip"] { color:#8B4A08; background:var(--tsa-orange-soft); }\n    [data-toposort-animation] .tsa-action-badge[data-kind="append"] { color:#1F655F; background:var(--tsa-teal-soft); }\n    [data-toposort-animation] .tsa-action-badge[data-kind="unwind"] { color:#4D6367; background:#E9EEEE; }\n    [data-toposort-animation] .tsa-action-badge[data-kind="return"] { color:#235893; background:var(--tsa-blue-soft); }\n    [data-toposort-animation] .tsa-action-badge[data-kind="reverse"] { color:#8B4A08; background:var(--tsa-orange-soft); }\n    [data-toposort-animation] .tsa-status { color:var(--tsa-ink); font-size:.94rem; line-height:1.46; }\n    [data-toposort-animation] .tsa-status strong { color:inherit; }\n    [data-toposort-animation] .tsa-graph-scroll { max-width:100%; overflow-x:auto; margin-top:12px; padding-bottom:3px; }\n    [data-toposort-animation] .tsa-graph-scroll:focus-visible { outline:3px solid rgba(43,108,176,.3); outline-offset:2px; }\n    [data-toposort-animation] svg.tsa-graph { display:block; width:100%; height:auto; min-width:690px; }\n    [data-toposort-animation] .tsa-edge {\n      fill:none; stroke:#71878A; stroke-width:2.2; marker-end:url(#tsa-forward-arrow);\n    }\n    [data-toposort-animation] .tsa-edge.is-active {\n      stroke:var(--tsa-orange); stroke-width:5; stroke-dasharray:9 7;\n      marker-start:url(#tsa-parent-arrow); marker-end:none; animation:tsa-dash .75s linear infinite;\n    }\n    [data-toposort-animation] .tsa-node rect { fill:#FFFFFF; stroke:#71878A; stroke-width:1.8; transition:fill .16s,stroke .16s,stroke-width .16s; }\n    [data-toposort-animation] .tsa-node text:first-of-type { fill:var(--tsa-ink); font-size:20px; font-weight:820; }\n    [data-toposort-animation] .tsa-node text:last-of-type { fill:#60777B; font-size:11px; font-weight:650; }\n    [data-toposort-animation] .tsa-node.is-seen rect { fill:var(--tsa-blue-soft); stroke:var(--tsa-blue); }\n    [data-toposort-animation] .tsa-node.is-appended rect { fill:var(--tsa-teal-soft); stroke:var(--tsa-teal); }\n    [data-toposort-animation] .tsa-node.is-stack rect { stroke:var(--tsa-blue); stroke-width:3; }\n    [data-toposort-animation] .tsa-node.is-current rect { stroke:#EB811B; stroke-width:4; }\n    [data-toposort-animation] .tsa-node.is-link-target rect { stroke:var(--tsa-orange); stroke-width:4; }\n    [data-toposort-animation] .tsa-node-index { fill:#52696D; font-size:10px; font-weight:750; }\n    [data-toposort-animation] .tsa-direction { margin:6px 0 0; color:var(--tsa-muted); font-size:.78rem; }\n    [data-toposort-animation] .tsa-node-legend { display:flex; flex-wrap:wrap; gap:8px 14px; margin-top:8px; color:var(--tsa-muted); font-size:.75rem; }\n    [data-toposort-animation] .tsa-node-legend span { display:inline-flex; align-items:center; gap:6px; }\n    [data-toposort-animation] .tsa-swatch { width:13px; height:13px; border:2px solid #71878A; border-radius:4px; background:#FFF; }\n    [data-toposort-animation] .tsa-swatch.seen { background:var(--tsa-blue-soft); border-color:var(--tsa-blue); }\n    [data-toposort-animation] .tsa-swatch.stack { border:3px solid var(--tsa-blue); }\n    [data-toposort-animation] .tsa-swatch.appended { background:var(--tsa-teal-soft); border-color:var(--tsa-teal); }\n    [data-toposort-animation] .tsa-machine { padding:16px; min-width:0; background:#FBFCFC; }\n    [data-toposort-animation] .tsa-panel-title { margin:0 0 8px; font-size:.78rem; color:var(--tsa-muted); font-weight:820; letter-spacing:.05em; text-transform:uppercase; }\n    [data-toposort-animation] .tsa-code { margin:0 0 16px; padding:9px 0; overflow-x:auto; border-radius:11px; background:#20363B; color:#EAF2F2; font:12px/1.58 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace; }\n    [data-toposort-animation] .tsa-code-line { display:block; padding:0 11px; white-space:pre; border-left:3px solid transparent; }\n    [data-toposort-animation] .tsa-code-line.is-active { background:#314E55; border-left-color:#F1A34B; color:#FFFFFF; }\n    [data-toposort-animation] .tsa-state-box { margin-top:12px; }\n    [data-toposort-animation] .tsa-link-state {\n      min-height:68px; padding:10px; border:1px solid #D8E1E2; border-radius:10px;\n      background:#FFFFFF; color:var(--tsa-ink); font-size:.82rem; line-height:1.5;\n    }\n    [data-toposort-animation] .tsa-link-state strong { color:var(--tsa-orange); }\n    [data-toposort-animation] .tsa-link-local { display:block; margin-top:4px; color:#52696D; }\n    [data-toposort-animation] .tsa-state-note { margin:5px 0 0; color:#52696D; font-size:.75rem; line-height:1.4; }\n    [data-toposort-animation] .tsa-chip-row { display:flex; flex-wrap:wrap; align-items:center; gap:6px; min-height:36px; }\n    [data-toposort-animation] .tsa-chip { display:inline-flex; align-items:center; justify-content:center; min-width:31px; min-height:31px; padding:4px 8px; border-radius:8px; color:var(--tsa-ink); background:#FFFFFF; border:1px solid #71878A; font:750 .84rem ui-monospace,SFMono-Regular,Menlo,monospace; }\n    [data-toposort-animation] .tsa-chip.stack { color:#235893; border:2px solid var(--tsa-blue); background:#F8FBFF; }\n    [data-toposort-animation] .tsa-chip.seen { color:#235893; border-color:#7EA5D1; background:var(--tsa-blue-soft); }\n    [data-toposort-animation] .tsa-chip.output { color:#1F655F; border-color:#79B8B2; background:var(--tsa-teal-soft); }\n    [data-toposort-animation] .tsa-chip-arrow { color:#52696D; font-weight:800; }\n    [data-toposort-animation] .tsa-empty { color:#52696D; font-size:.82rem; font-style:italic; }\n    [data-toposort-animation] .tsa-ledgers { display:grid; grid-template-columns:1fr 1fr; gap:12px; padding:14px 16px 16px; border-top:1px solid #E2E9EA; }\n    [data-toposort-animation] .tsa-ledger { min-width:0; padding:12px; border:1px solid #D8E1E2; border-radius:12px; background:#FFFFFF; }\n    [data-toposort-animation] .tsa-ledger p { margin:7px 0 0; color:var(--tsa-muted); font-size:.78rem; line-height:1.4; }\n    [data-toposort-animation] .tsa-backward { margin:0 16px 16px; padding:14px; border:1px solid #EB811B; border-radius:13px; background:var(--tsa-orange-soft); }\n    [data-toposort-animation] .tsa-backward[hidden] { display:none; }\n    [data-toposort-animation] .tsa-backward .tsa-chip { background:#FFFFFF; border-color:#D88B3D; color:#7B430B; }\n    [data-toposort-animation] .tsa-backward p { margin:9px 0 0; color:#5E421F; font-size:.88rem; line-height:1.48; }\n    [data-toposort-animation] .tsa-action-key { display:flex; flex-wrap:wrap; gap:6px 10px; padding:11px 16px 13px; border-top:1px solid #E2E9EA; color:var(--tsa-muted); font-size:.72rem; }\n    [data-toposort-animation] .tsa-action-key span { display:inline-flex; align-items:center; gap:5px; }\n    [data-toposort-animation] .tsa-legend-badge { min-width:auto; padding:4px 7px; }\n    [data-toposort-animation] .tsa-legend-badge.enter { color:#235893; background:var(--tsa-blue-soft); }\n    [data-toposort-animation] .tsa-legend-badge.follow { color:#8B4A08; background:var(--tsa-orange-soft); }\n    [data-toposort-animation] .tsa-legend-badge.skip { color:#8B4A08; background:var(--tsa-orange-soft); }\n    [data-toposort-animation] .tsa-legend-badge.append { color:#1F655F; background:var(--tsa-teal-soft); }\n    [data-toposort-animation] .tsa-legend-badge.unwind { color:#4D6367; background:#E9EEEE; }\n    [data-toposort-animation] .tsa-legend-badge.return { color:#235893; background:var(--tsa-blue-soft); }\n    [data-toposort-animation] .tsa-legend-badge.reverse { color:#8B4A08; background:var(--tsa-orange-soft); }\n    [data-toposort-animation] .tsa-keyboard { width:100%; margin-top:2px; color:#52696D; }\n    [data-toposort-animation] .tsa-print-summary { display:none; }\n    @keyframes tsa-dash { to { stroke-dashoffset:-16; } }\n    @media (max-width:850px) {\n      [data-toposort-animation] .tsa-main { grid-template-columns:1fr; }\n      [data-toposort-animation] .tsa-visual { border-right:0; border-bottom:1px solid #E2E9EA; }\n      [data-toposort-animation] .tsa-ledgers { grid-template-columns:1fr; }\n      [data-toposort-animation] .tsa-step-count { width:100%; margin-left:0; }\n    }\n    @media (max-width:520px) {\n      [data-toposort-animation] .tsa-heading { padding:17px 15px 14px; }\n      [data-toposort-animation] .tsa-controls { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); padding:10px; gap:6px; }\n      [data-toposort-animation] .tsa-speed, [data-toposort-animation] .tsa-step-count { grid-column:1 / -1; margin-left:0; }\n      [data-toposort-animation] .tsa-visual, [data-toposort-animation] .tsa-machine { padding:12px; }\n      [data-toposort-animation] .tsa-ledgers { padding:12px; }\n    }\n    @media (prefers-reduced-motion:reduce) {\n      [data-toposort-animation] .tsa-edge.is-active { animation:none; }\n      [data-toposort-animation] .tsa-node rect { transition:none; }\n    }\n    @media print {\n      [data-toposort-animation] .tsa-heading,\n      [data-toposort-animation] .tsa-controls,\n      [data-toposort-animation] .tsa-main,\n      [data-toposort-animation] .tsa-ledgers,\n      [data-toposort-animation] .tsa-backward,\n      [data-toposort-animation] .tsa-action-key { display:none !important; }\n      [data-toposort-animation] .tsa-print-summary {\n        display:block; padding:18px 22px; color:var(--tsa-ink); line-height:1.6;\n      }\n    }\n  </style>\n\n  <div class="tsa-shell">\n    <header class="tsa-heading">\n      <p class="tsa-kicker">Interactive trace · exact notebook graph</p>\n      <h4>Watch <code>dependency_safe_order(L)</code> build its list</h4>\n      <p class="tsa-intro">Each step is one action performed by the recursive helper. The graph is fixed;\n        only the call stack, <code>seen</code>, and <code>safe_order</code> change.</p>\n    </header>\n\n    <div class="tsa-controls" role="group" aria-label="Animation controls">\n      <button type="button" data-action="reset" title="Reset (Home)">Reset</button>\n      <button type="button" data-action="previous" title="Previous step (Left arrow)">Previous</button>\n      <button type="button" data-action="play" aria-pressed="false" title="Play or pause (Space)">Play</button>\n      <button type="button" data-action="next" title="Next step (Right arrow)">Next</button>\n      <label class="tsa-speed">Speed\n        <select data-action="speed" aria-label="Playback speed">\n          <option value="1500">0.6×</option>\n          <option value="950" selected>1×</option>\n          <option value="520">1.8×</option>\n        </select>\n      </label>\n      <span class="tsa-step-count" data-step-count>Step 0 of 0</span>\n    </div>\n\n    <div class="tsa-main">\n      <div class="tsa-visual">\n        <div class="tsa-action">\n          <span class="tsa-action-badge" data-kind="ready" data-action-badge>Ready</span>\n          <div class="tsa-status" data-status role="status" aria-live="polite">Press <strong>Next</strong> or <strong>Play</strong> to call <code>append_after_parents(L)</code>.</div>\n        </div>\n\n        <div class="tsa-graph-scroll" tabindex="0" role="region"\n             aria-label="Scrollable computation graph; use left and right arrow keys to pan">\n          <svg class="tsa-graph" viewBox="0 0 900 350" role="img">\n            <title>Computation graph used by the dependency-order trace</title>\n            <desc>The forward graph has leaves w and x creating m, m and b creating a, a and y creating e, and e creating L. The recursive traversal follows those edges in reverse, from each output to its parent operands.</desc>\n            <defs>\n              <marker id="tsa-forward-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto">\n                <path d="M 0 0 L 10 5 L 0 10 z" fill="#71878A"/>\n              </marker>\n              <marker id="tsa-parent-arrow" viewBox="0 0 10 10" refX="1" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">\n                <path d="M 0 0 L 10 5 L 0 10 z" fill="#B35F0B"/>\n              </marker>\n            </defs>\n\n            <path class="tsa-edge" data-child="m" data-parent="w" d="M150 62 C168 62 173 104 190 112"/>\n            <path class="tsa-edge" data-child="m" data-parent="x" d="M150 157 C168 157 173 120 190 112"/>\n            <path class="tsa-edge" data-child="a" data-parent="m" d="M315 112 C340 112 350 156 375 167"/>\n            <path class="tsa-edge" data-child="a" data-parent="b" d="M315 237 C340 237 350 178 375 167"/>\n            <path class="tsa-edge" data-child="e" data-parent="a" d="M500 167 C525 167 535 216 560 227"/>\n            <path class="tsa-edge" data-child="e" data-parent="y" d="M500 292 C525 292 535 238 560 227"/>\n            <path class="tsa-edge" data-child="L" data-parent="e" d="M685 227 L745 227"/>\n\n            <g class="tsa-node" data-node="w" transform="translate(25 30)"><rect width="125" height="64" rx="13"/><text x="62.5" y="27" text-anchor="middle">w</text><text x="62.5" y="48" text-anchor="middle">leaf · value 2</text></g>\n            <g class="tsa-node" data-node="x" transform="translate(25 125)"><rect width="125" height="64" rx="13"/><text x="62.5" y="27" text-anchor="middle">x</text><text x="62.5" y="48" text-anchor="middle">leaf · value 3</text></g>\n            <g class="tsa-node" data-node="m" transform="translate(190 80)"><rect width="125" height="64" rx="13"/><text x="62.5" y="27" text-anchor="middle">m</text><text x="62.5" y="48" text-anchor="middle">w × x = 6</text></g>\n            <g class="tsa-node" data-node="b" transform="translate(190 205)"><rect width="125" height="64" rx="13"/><text x="62.5" y="27" text-anchor="middle">b</text><text x="62.5" y="48" text-anchor="middle">leaf · value 1</text></g>\n            <g class="tsa-node" data-node="a" transform="translate(375 135)"><rect width="125" height="64" rx="13"/><text x="62.5" y="27" text-anchor="middle">a</text><text x="62.5" y="48" text-anchor="middle">m + b = 7</text></g>\n            <g class="tsa-node" data-node="y" transform="translate(375 260)"><rect width="125" height="64" rx="13"/><text x="62.5" y="27" text-anchor="middle">y</text><text x="62.5" y="48" text-anchor="middle">leaf · value 10</text></g>\n            <g class="tsa-node" data-node="e" transform="translate(560 195)"><rect width="125" height="64" rx="13"/><text x="62.5" y="27" text-anchor="middle">e</text><text x="62.5" y="48" text-anchor="middle">a − y = −3</text></g>\n            <g class="tsa-node" data-node="L" transform="translate(745 195)"><rect width="125" height="64" rx="13"/><text x="62.5" y="27" text-anchor="middle">L</text><text x="62.5" y="48" text-anchor="middle">e² = 9</text></g>\n\n            <text x="25" y="342" class="tsa-node-index">forward graph: operands → outputs &nbsp; · &nbsp; traversal: output → saved parent operand</text>\n          </svg>\n        </div>\n        <p class="tsa-direction">The moving dashed edge is a saved <code>ParentLink</code> being followed from an output back to one direct operand. On a phone, scroll the graph sideways.</p>\n        <div class="tsa-node-legend" aria-label="Node color key">\n          <span><i class="tsa-swatch"></i>unseen</span>\n          <span><i class="tsa-swatch seen"></i>seen</span>\n          <span><i class="tsa-swatch stack"></i>on call stack</span>\n          <span><i class="tsa-swatch appended"></i>appended</span>\n        </div>\n      </div>\n\n      <aside class="tsa-machine" aria-label="Algorithm state">\n        <p class="tsa-panel-title">The code now running</p>\n        <pre class="tsa-code" aria-label="dependency safe order pseudocode"><code><span class="tsa-code-line" data-line="1">def append_after_parents(node):</span><span class="tsa-code-line" data-line="2">  if id(node) in seen: return</span><span class="tsa-code-line" data-line="3">  seen.add(id(node))</span><span class="tsa-code-line" data-line="4">  for link in node.parents:</span><span class="tsa-code-line" data-line="5">    append_after_parents(link.value)</span><span class="tsa-code-line" data-line="6">  safe_order.append(node)</span><span class="tsa-code-line" data-line="7"></span><span class="tsa-code-line" data-line="8">append_after_parents(L)</span><span class="tsa-code-line" data-line="9">return safe_order</span><span class="tsa-code-line" data-line="10"></span><span class="tsa-code-line" data-line="11">for output in reversed(safe_order):  # backward</span></code></pre>\n\n        <div class="tsa-state-box">\n          <p class="tsa-panel-title">Active recursive calls</p>\n          <div class="tsa-chip-row" data-stack aria-label="Call stack"><span class="tsa-empty">empty</span></div>\n        </div>\n        <div class="tsa-state-box">\n          <p class="tsa-panel-title">Seen Values</p>\n          <div class="tsa-chip-row" data-seen aria-label="Seen Values"><span class="tsa-empty">none yet</span></div>\n          <p class="tsa-state-note">Shown in discovery order for readability; <code>seen</code> is a set, so only membership matters.</p>\n        </div>\n        <div class="tsa-state-box">\n          <p class="tsa-panel-title">Current ParentLink</p>\n          <div class="tsa-link-state" data-link-state aria-live="polite"><span class="tsa-empty">No link is being followed.</span></div>\n          <p class="tsa-state-note">Sorting follows <code>.value</code>. The saved <code>.local_grad</code> is displayed but remains unused until backward.</p>\n        </div>\n      </aside>\n    </div>\n\n    <div class="tsa-ledgers">\n      <section class="tsa-ledger">\n        <p class="tsa-panel-title">Dependency-first output being built</p>\n        <div class="tsa-chip-row" data-order aria-label="Dependency-first output"><span class="tsa-empty">empty</span></div>\n        <p>A Value enters this list only after every recursive parent call has returned.</p>\n      </section>\n      <section class="tsa-ledger">\n        <p class="tsa-panel-title">What the current structures mean</p>\n        <p><code>stack</code> answers “which calls are waiting?” · <code>seen</code> prevents scheduling one shared Value twice · <code>safe_order</code> is the list we will reverse.</p>\n      </section>\n    </div>\n\n    <section class="tsa-backward" data-backward hidden aria-label="Final backward schedule">\n      <p class="tsa-panel-title">Reverse once · backward schedule</p>\n      <div class="tsa-chip-row" data-backward-order></div>\n      <p><strong>Why every node is ready:</strong> reversing puts an output before the operands it can update.\n        Therefore every later output that can contribute to a node\'s <code>.grad</code> has already run before\n        that node sends its completed gradient to its own parents.</p>\n    </section>\n\n    <div class="tsa-action-key" aria-label="Action type key">\n      <span><b class="tsa-legend-badge enter">enter</b> start one call and mark new</span>\n      <span><b class="tsa-legend-badge follow">follow link</b> recurse to an operand</span>\n      <span><b class="tsa-legend-badge skip">skip seen</b> immediate return for a shared Value; this exact graph never needs it</span>\n      <span><b class="tsa-legend-badge append">append</b> parents are finished</span>\n      <span><b class="tsa-legend-badge unwind">unwind</b> child call returned</span>\n      <span><b class="tsa-legend-badge return">return</b> sorting is finished</span>\n      <span><b class="tsa-legend-badge reverse">reverse</b> enter the backward schedule</span>\n      <span class="tsa-keyboard">Keyboard when the panel itself is focused: ← previous · → next · Space play/pause · Home reset.</span>\n    </div>\n    <div class="tsa-print-summary">\n      <strong>Dependency-first order:</strong> w → x → m → b → a → y → e → L<br>\n      <strong>Reverse for backward:</strong> L → e → y → a → b → m → x → w<br>\n      Each operand appears before the output that uses it; reversing makes each output ready before it sends gradient contributions to its operands.\n    </div>\n  </div>\n  <noscript><p><strong>JavaScript is off.</strong> Use the static dependency-order diagram immediately above this trace.</p></noscript>\n</section>\n<script data-toposort-animation-init>\n(() => {\n  const payload = __TOPOLOGY_PAYLOAD__;\n  const script = document.currentScript;\n  const root = script && script.previousElementSibling;\n  if (!root || !root.matches("[data-toposort-animation]") || root.dataset.enhanced === "true") return;\n  root.dataset.enhanced = "true";\n  const instanceId = "toposort-animation";\n  root.id = instanceId;\n  const svg = root.querySelector("svg.tsa-graph");\n  const svgTitle = svg.querySelector("title");\n  const svgDesc = svg.querySelector("desc");\n  svgTitle.id = `${instanceId}-graph-title`;\n  svgDesc.id = `${instanceId}-graph-desc`;\n  svg.setAttribute("aria-labelledby", `${svgTitle.id} ${svgDesc.id}`);\n\n  const events = payload.events;\n\n  const labels = {\n    ready:"Ready", enter:"Enter", follow:"Follow link", skip:"Skip seen",\n    append:"Append", unwind:"Unwind", return:"Return", reverse:"Reverse"\n  };\n  const buttons = {\n    reset:root.querySelector(\'[data-action="reset"]\'),\n    previous:root.querySelector(\'[data-action="previous"]\'),\n    play:root.querySelector(\'[data-action="play"]\'),\n    next:root.querySelector(\'[data-action="next"]\')\n  };\n  const speed = root.querySelector(\'[data-action="speed"]\');\n  const actionBadge = root.querySelector("[data-action-badge]");\n  const status = root.querySelector("[data-status]");\n  const stepCount = root.querySelector("[data-step-count]");\n  const backwardPanel = root.querySelector("[data-backward]");\n  const linkState = root.querySelector("[data-link-state]");\n  const nodeElements = [...root.querySelectorAll("[data-node]")];\n  const edgeElements = [...root.querySelectorAll(".tsa-edge")];\n  const codeLines = [...root.querySelectorAll("[data-line]")];\n  let index = 0;\n  let timer = null;\n  let playing = false;\n  let lastReportedHeight = 0;\n\n  function reportHeight() {\n    const height = Math.ceil(root.getBoundingClientRect().height + 16);\n    if (height === lastReportedHeight) return;\n    lastReportedHeight = height;\n    window.parent.postMessage({type:"scalar-topology-animation-height", height}, "*");\n  }\n\n  function chips(target, values, kind, emptyText) {\n    target.replaceChildren();\n    if (!values.length) {\n      const empty = document.createElement("span");\n      empty.className = "tsa-empty";\n      empty.textContent = emptyText;\n      target.append(empty);\n      return;\n    }\n    values.forEach((value, position) => {\n      if (position) {\n        const arrow = document.createElement("span");\n        arrow.className = "tsa-chip-arrow";\n        arrow.textContent = "→";\n        arrow.setAttribute("aria-hidden", "true");\n        target.append(arrow);\n      }\n      const chip = document.createElement("span");\n      chip.className = `tsa-chip ${kind}`;\n      chip.textContent = value;\n      target.append(chip);\n    });\n  }\n\n  function stop() {\n    if (timer !== null) window.clearTimeout(timer);\n    timer = null;\n    playing = false;\n    buttons.play.textContent = "Play";\n    buttons.play.setAttribute("aria-pressed", "false");\n  }\n\n  function render() {\n    const event = events[index];\n    const backwardOrder = [...event.order].reverse();\n    actionBadge.dataset.kind = event.kind;\n    actionBadge.textContent = labels[event.kind];\n    status.innerHTML = event.message;\n    stepCount.textContent = `Step ${index} of ${events.length - 1}`;\n    buttons.previous.disabled = index === 0;\n    buttons.next.disabled = index === events.length - 1;\n\n    codeLines.forEach(line => line.classList.toggle("is-active", Number(line.dataset.line) === event.line));\n    nodeElements.forEach(nodeElement => {\n      const name = nodeElement.dataset.node;\n      nodeElement.classList.toggle("is-seen", event.seen.includes(name));\n      nodeElement.classList.toggle("is-stack", event.stack.includes(name));\n      nodeElement.classList.toggle("is-appended", event.order.includes(name));\n      nodeElement.classList.toggle("is-current", event.node === name && event.kind !== "follow");\n      nodeElement.classList.toggle("is-link-target", event.kind === "follow" && event.parent === name);\n    });\n    edgeElements.forEach(edge => edge.classList.toggle(\n      "is-active",\n      event.kind === "follow" && edge.dataset.child === event.node && edge.dataset.parent === event.parent\n    ));\n\n    linkState.replaceChildren();\n    if (event.kind === "follow") {\n      const linkName = document.createElement("code");\n      linkName.textContent = `${event.node}.parents[${event.link_index}]`;\n      const valueLine = document.createElement("strong");\n      valueLine.style.display = "block";\n      valueLine.textContent = `.value = ${event.parent}  ← followed now`;\n      const localLine = document.createElement("span");\n      localLine.className = "tsa-link-local";\n      localLine.textContent = `.local_grad = ${event.local_grad}  · stored, not read by sorting`;\n      linkState.append(linkName, valueLine, localLine);\n    } else {\n      const empty = document.createElement("span");\n      empty.className = "tsa-empty";\n      empty.textContent = "No link is being followed in this step.";\n      linkState.append(empty);\n    }\n\n    chips(root.querySelector("[data-stack]"), event.stack, "stack", "empty");\n    chips(root.querySelector("[data-seen]"), event.seen, "seen", "none yet");\n    chips(root.querySelector("[data-order]"), event.order, "output", "empty");\n    chips(root.querySelector("[data-backward-order]"), backwardOrder, "backward", "empty");\n    backwardPanel.hidden = event.kind !== "reverse";\n    window.requestAnimationFrame(reportHeight);\n  }\n\n  function scheduleNext() {\n    if (!playing) return;\n    timer = window.setTimeout(() => {\n      if (index < events.length - 1) {\n        index += 1;\n        render();\n        scheduleNext();\n      } else {\n        stop();\n      }\n    }, Number(speed.value));\n  }\n\n  function togglePlay() {\n    if (playing) {\n      stop();\n      return;\n    }\n    if (index === events.length - 1) index = 0;\n    playing = true;\n    buttons.play.textContent = "Pause";\n    buttons.play.setAttribute("aria-pressed", "true");\n    render();\n    scheduleNext();\n  }\n\n  buttons.reset.addEventListener("click", () => { stop(); index = 0; render(); });\n  buttons.previous.addEventListener("click", () => { stop(); index = Math.max(0, index - 1); render(); });\n  buttons.next.addEventListener("click", () => { stop(); index = Math.min(events.length - 1, index + 1); render(); });\n  buttons.play.addEventListener("click", togglePlay);\n  speed.addEventListener("change", () => {\n    if (playing) { window.clearTimeout(timer); scheduleNext(); }\n  });\n  document.addEventListener("visibilitychange", () => {\n    if (document.hidden) stop();\n  });\n  window.addEventListener("resize", () => window.requestAnimationFrame(reportHeight));\n  root.addEventListener("keydown", event => {\n    if (event.target !== root) return;\n    if (event.key === "ArrowRight") { event.preventDefault(); buttons.next.click(); }\n    else if (event.key === "ArrowLeft") { event.preventDefault(); buttons.previous.click(); }\n    else if (event.key === "Home") { event.preventDefault(); buttons.reset.click(); }\n    else if (event.key === " ") { event.preventDefault(); togglePlay(); }\n  });\n\n  render();\n})();\n</script>\n</body>\n</html>'

if importlib.util.find_spec("graphviz") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "graphviz"],
        check=True,
    )

from graphviz import Digraph
from IPython.display import HTML, display

def draw_graph(root, show_grad=True, min_width=780):
    nodes = dependency_safe_order(root)

    dot = Digraph(format="svg")
    dot.attr(rankdir="LR", bgcolor="transparent", pad="0.15",
             nodesep="0.25", ranksep="0.45")
    dot.attr("node", fontname="Helvetica", fontsize="11")
    dot.attr("edge", fontname="Helvetica", fontsize="9", color="#60777b")

    for node in nodes:
        node_id = "value_" + node.label
        grad = f"{node.grad:g}" if show_grad else "—"
        fill = "#e8f7f5" if show_grad and node.grad != 0 else "#ffffff"
        border = "#2C7A7B" if show_grad and node.grad != 0 else "#1f3a40"
        dot.node(
            node_id,
            label=f"{{ {node.label} | value {node.data:g} | grad {grad} }}",
            shape="record", style="rounded,filled", fillcolor=fill,
            color=border, fontcolor="#1f3a40",
        )
        if node.op:
            op_id = "op_" + node.label
            dot.node(op_id, label=node.op, shape="circle",
                     width="0.32", height="0.32", margin="0.03",
                     style="filled", fillcolor="#2B6CB0",
                     color="#2B6CB0", fontcolor="white")
            dot.edge(op_id, node_id)
            for link in node.parents:
                local_label = (
                    f"∂{node.label}/∂{link.value.label}="
                    f"{link.local_grad:g}"
                )
                dot.edge(
                    "value_" + link.value.label,
                    op_id,
                    label=local_label,
                    fontcolor="#2B6CB0",
                )
    svg = dot.pipe(format="svg").decode("utf-8")
    svg = svg.replace(
        "<svg ",
        f'<svg style="width:100%;height:auto;min-width:{min_width}px" ',
        1,
    )
    return HTML(
        '<div style="max-width:100%;overflow-x:auto">'
        f'<div style="min-width:{min_width}px">' + svg + '</div></div>'
    )

def trace_dependency_safe_order(root):
    """Run the real append-after-parents recursion while recording display states."""
    safe_order = []
    seen_ids = set()
    seen_values = []
    call_stack = []
    events = []

    def snapshot(
        kind, *, node=None, parent=None, link_index=None,
        local_grad=None, line, message,
    ):
        event = {
            "kind": kind,
            "stack": [value.label for value in call_stack],
            "seen": [value.label for value in seen_values],
            "order": [value.label for value in safe_order],
            "line": line,
            "message": message,
        }
        if node is not None:
            event["node"] = node.label
        if parent is not None:
            event["parent"] = parent.label
        if link_index is not None:
            event["link_index"] = link_index
        if local_grad is not None:
            event["local_grad"] = float(local_grad)
        events.append(event)

    snapshot(
        "ready", line=8,
        message=(
            "Press <strong>Next</strong> or <strong>Play</strong> to call "
            "<code>append_after_parents(L)</code>."
        ),
    )

    def append_after_parents(node):
        call_stack.append(node)
        label = escape_html(node.label)

        if id(node) in seen_ids:
            snapshot(
                "skip", node=node, line=2,
                message=(
                    f"<strong>{label}</strong> is already in <code>seen</code>, so this repeated "
                    "call returns without appending it again."
                ),
            )
            call_stack.pop()
            return

        seen_ids.add(id(node))
        seen_values.append(node)
        parent_count = len(node.parents)
        if parent_count:
            remaining = "its parent" if parent_count == 1 else f"all {parent_count} parents"
            enter_message = (
                f"Enter <strong>{label}</strong>. It is new, so mark it seen; "
                f"this call must now visit {remaining}."
            )
        else:
            enter_message = (
                f"Enter leaf <strong>{label}</strong>. It is new, so mark it seen. "
                "It has no ParentLinks, so it can be appended next."
            )
        snapshot("enter", node=node, line=3, message=enter_message)

        for link_index, link in enumerate(node.parents):
            parent = link.value
            parent_label = escape_html(parent.label)
            snapshot(
                "follow", node=node, parent=parent,
                link_index=link_index, local_grad=link.local_grad, line=5,
                message=(
                    f"Follow <code>{label}.parents[{link_index}]</code> to "
                    f"<strong>{parent_label}</strong>, then call "
                    f"<code>append_after_parents({parent_label})</code>."
                ),
            )
            append_after_parents(parent)
            if link_index + 1 < parent_count:
                next_action = "Continue to the next ParentLink."
            else:
                next_action = "All of this node's parent calls are now finished."
            snapshot(
                "unwind", node=node, parent=parent, line=5,
                message=(
                    f"The call for <strong>{parent_label}</strong> has returned to "
                    f"<strong>{label}</strong>. {next_action}"
                ),
            )

        safe_order.append(node)
        snapshot(
            "append", node=node, line=6,
            message=(
                f"Append <strong>{label}</strong> to <code>safe_order</code>. Every direct "
                f"parent of {label} is already earlier in the list."
            ),
        )
        call_stack.pop()

    append_after_parents(root)
    snapshot(
        "return", line=9,
        message=(
            "The recursive helper is finished. Return the dependency-first "
            "<code>safe_order</code> unchanged."
        ),
    )
    snapshot(
        "reverse", line=11,
        message=(
            "Now move into <code>backward</code>. Its loop reads "
            "<code>reversed(safe_order)</code> before performing any local-gradient arithmetic."
        ),
    )

    # This trace must be a faithful observation of the actual helper above it.
    assert safe_order == dependency_safe_order(root)
    assert events[-1]["order"] == [node.label for node in safe_order]
    return events

def show_topological_sort_animation(root, events):
    """Render the recorded traversal in an isolated, deterministic iframe."""
    nodes = dependency_safe_order(root)
    order_labels = [node.label for node in nodes]
    backward_labels = list(reversed(order_labels))
    parent_labels = {
        node.label: [link.value.label for link in node.parents]
        for node in nodes
    }
    node_data = {node.label: node.data for node in nodes}
    node_ops = {node.label: node.op for node in nodes}
    link_state = {
        node.label: [
            (link.value.label, float(link.local_grad))
            for link in node.parents
        ]
        for node in nodes
    }

    # The layout below is purpose-built for this lecture's exact graph.
    expected_order = ["w", "x", "m", "b", "a", "y", "e", "L"]
    expected_backward = ["L", "e", "y", "a", "b", "m", "x", "w"]
    expected_parents = {
        "w": [], "x": [], "m": ["w", "x"], "b": [],
        "a": ["m", "b"], "y": [], "e": ["a", "y"], "L": ["e"],
    }
    expected_data = {
        "w": 2.0, "x": 3.0, "m": 6.0, "b": 1.0,
        "a": 7.0, "y": 10.0, "e": -3.0, "L": 9.0,
    }
    expected_ops = {
        "w": "", "x": "", "m": "×", "b": "",
        "a": "+", "y": "", "e": "−", "L": "²",
    }
    expected_links = {
        "w": [], "x": [], "m": [("w", 3.0), ("x", 2.0)], "b": [],
        "a": [("m", 1.0), ("b", 1.0)], "y": [],
        "e": [("a", 1.0), ("y", -1.0)], "L": [("e", -6.0)],
    }
    assert order_labels == expected_order
    assert backward_labels == expected_backward
    assert parent_labels == expected_parents
    assert node_data == expected_data
    assert node_ops == expected_ops
    assert link_state == expected_links
    assert events[0]["kind"] == "ready"
    assert events[-2]["kind"] == "return"
    assert events[-1]["kind"] == "reverse"
    assert events[-1]["order"] == expected_order
    assert sum(event["kind"] == "enter" for event in events) == 8
    assert sum(event["kind"] == "follow" for event in events) == 7
    assert sum(event["kind"] == "unwind" for event in events) == 7
    assert sum(event["kind"] == "append" for event in events) == 8
    assert sum(event["kind"] == "return" for event in events) == 1
    assert sum(event["kind"] == "reverse" for event in events) == 1
    assert not any(event["kind"] == "skip" for event in events)

    # Audit every recorded intermediate state, not only the final list.
    previous_seen = []
    previous_order = []
    for event in events:
        seen = event["seen"]
        order = event["order"]
        stack = event["stack"]
        assert seen[:len(previous_seen)] == previous_seen
        assert order[:len(previous_order)] == previous_order
        assert len(seen) == len(set(seen))
        assert len(order) == len(set(order))
        assert set(order) <= set(seen)
        assert set(stack) <= set(seen)
        for output_label, parent_label in zip(stack, stack[1:]):
            assert parent_label in parent_labels[output_label]

        if event["kind"] == "enter":
            assert stack[-1] == event["node"] == seen[-1]
        elif event["kind"] == "follow":
            assert stack[-1] == event["node"]
            index = event["link_index"]
            assert 0 <= index < len(link_state[event["node"]])
            expected_parent, expected_local = link_state[event["node"]][index]
            assert event["parent"] == expected_parent
            assert event["local_grad"] == expected_local
        elif event["kind"] == "append":
            assert stack[-1] == event["node"] == order[-1]
            assert set(parent_labels[event["node"]]) <= set(order[:-1])
        elif event["kind"] == "unwind":
            assert stack[-1] == event["node"]
            assert event["parent"] in parent_labels[event["node"]]
        elif event["kind"] in {"return", "reverse"}:
            assert not stack and order == expected_order

        previous_seen = seen
        previous_order = order

    payload = {
        "events": events,
        "parents": parent_labels,
        "order": order_labels,
        "backward": backward_labels,
    }
    marker = "__TOPOLOGY_PAYLOAD__"
    assert TOPOLOGY_SORT_ANIMATION_DOCUMENT.count(marker) == 1
    document = TOPOLOGY_SORT_ANIMATION_DOCUMENT.replace(
        marker,
        json.dumps(payload, ensure_ascii=False, separators=(",", ":")),
    )
    iframe = (
        "<style>"
        ".scalar-topology-animation-frame{height:1120px}"
        "@media(max-width:850px){.scalar-topology-animation-frame{height:1750px}}"
        "@media(max-width:520px){.scalar-topology-animation-frame{height:2360px}}"
        "@media print{.scalar-topology-animation-frame{height:180px!important}}"
        "</style>"
        '<iframe class="scalar-topology-animation-frame" '
        'title="Interactive dependency-safe ordering trace" sandbox="allow-scripts" '
        'style="display:block;width:100%;border:0;margin:14px 0 20px;" '
        f'srcdoc="{escape_html(document, quote=True)}"></iframe>'
        "<script>(()=>{"
        "const frame=document.currentScript.previousElementSibling;"
        "const resize=(event)=>{"
        "if(event.source!==frame.contentWindow||event.data?.type!=='scalar-topology-animation-height')return;"
        "const height=Math.max(800,Math.min(2600,Number(event.data.height)||1120));"
        "frame.style.height=`${height}px`;"
        "};"
        "window.addEventListener('message',resize);"
        "})();</script>"
    )
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Consider using IPython.display.IFrame instead",
            category=UserWarning,
        )
        display(HTML(iframe))

def show_backward_steps(steps, highlight_outputs=()):
    rows = []
    for number, step in enumerate(steps, start=1):
        output = step["output"]
        parent = step["parent"]
        focused = output in highlight_outputs
        border = "#2B6CB0" if focused else "#d6e2e1"
        background = "#f4f8ff" if focused else "#fff"
        rows.append(
            f"<div style='border:1px solid {border};border-radius:8px;"
            f"padding:10px 12px;margin:8px 0;background:{background}'>"
            f"<div style='font-weight:700;margin-bottom:4px'>{number}. {output} → {parent}</div>"
            "<div style='font-size:1.02em;line-height:1.55'>"
            f"<span style='color:#2C7A7B;font-weight:700'>g<sub>{output}</sub> = {step['upstream']:g}</span>"
            " &nbsp;×&nbsp; "
            f"<span style='color:#2B6CB0;font-weight:700'>∂{output}/∂{parent} = {step['local']:g}</span>"
            " &nbsp;=&nbsp; "
            f"<span style='color:#EB811B;font-weight:700'>Δg<sub>{parent}</sub> = {step['downstream']:g}</span>"
            "</div>"
            f"<div style='color:#526669;margin-top:3px'>{parent}.grad: "
            f"{step['before']:g} → {step['after']:g}</div></div>"
        )

    display(HTML(
        "<div style='border-left:5px solid #2C7A7B;background:#eef8f7;"
        "padding:10px 12px;margin:4px 0 12px;border-radius:4px'>"
        "<b>Seed:</b> L.grad = ∂L/∂L = 1</div>" + "".join(rows)
    ))

def show_complete_state(root):
    """Show every stored Value field and every saved ParentLink."""
    cards = []
    for node in dependency_safe_order(root):
        label = escape_html(node.label)
        label_value = escape_html(repr(node.label))
        op_value = escape_html(repr(node.op))
        op_note = "" if node.op else " &nbsp;—&nbsp; input / leaf"

        if node.parents:
            parent_rows = []
            for index, link in enumerate(node.parents, start=1):
                parent = escape_html(link.value.label)
                parent_rows.append(
                    "<div style='display:grid;grid-template-columns:auto 1fr;gap:5px 10px;"
                    "align-items:baseline;padding:7px 0;border-top:1px solid #e2e9e9'>"
                    f"<span style='color:#60777B;font-size:0.82rem'>.parents[{index - 1}]</span>"
                    f"<code style='font-weight:750'>ParentLink(value={parent})</code>"
                    "<span style='color:#60777B;font-size:0.82rem'>.value</span>"
                    f"<span><code>{parent}</code></span>"
                    "<span style='color:#60777B;font-size:0.82rem'>.local_grad</span>"
                    f"<span style='color:#2B6CB0;font-weight:750'>{link.local_grad:g}"
                    f" &nbsp; (= ∂{label}/∂{parent})</span></div>"
                )
            parents_html = "".join(parent_rows)
        else:
            parents_html = (
                "<div style='padding:9px 0 2px;color:#60777B;border-top:1px solid #e2e9e9'>"
                "<code>.parents = ()</code> &nbsp;—&nbsp; no direct operands</div>"
            )

        cards.append(
            "<section style='min-width:0;border:1px solid #cddada;border-radius:12px;"
            "background:#FFFFFF;overflow:hidden'>"
            "<div style='display:flex;justify-content:space-between;align-items:baseline;gap:12px;"
            "padding:10px 12px;background:#F4F7F7;border-bottom:1px solid #dce5e5'>"
            f"<strong style='font-size:1.2rem;color:#1F3A40'>Value {label}</strong>"
            f"<span style='color:#60777B;font-size:0.85rem'><code>.label = {label_value}</code></span></div>"
            "<div style='padding:10px 12px'>"
            "<div style='display:grid;grid-template-columns:auto 1fr;gap:7px 10px;align-items:baseline'>"
            "<span style='color:#60777B'><code>.data</code></span>"
            f"<strong style='color:#1F3A40'>{node.data:g}</strong>"
            "<span style='color:#60777B'><code>.grad</code></span>"
            f"<strong style='color:#2C7A7B'>{node.grad:g} "
            f"<span style='font-size:0.88rem;font-weight:650'>(= ∂L/∂{label})</span></strong>"
            "<span style='color:#60777B'><code>.op</code></span>"
            f"<strong style='color:#1F3A40'><code>{op_value}</code>{op_note}</strong>"
            "<span style='color:#60777B'><code>.parents</code></span>"
            f"<strong style='color:#1F3A40'>{len(node.parents)} saved link"
            f"{'s' if len(node.parents) != 1 else ''}</strong></div>"
            f"<div style='margin-top:9px'>{parents_html}</div></div></section>"
        )

    order_text = " → ".join(escape_html(node.label) for node in dependency_safe_order(root))
    display(HTML(
        "<div style='margin:18px 0 8px'>"
        "<div style='border-left:5px solid #2C7A7B;background:#EEF8F7;"
        "padding:10px 12px;margin-bottom:12px;border-radius:4px'>"
        "<strong>Complete stored state after backward</strong><br>"
        "<span style='color:#526669'>The graph above stays compact. These cards expose every "
        "<code>Value</code> field and every saved <code>ParentLink</code>. "
        "The orange edge contribution is not a <code>Value</code> or <code>ParentLink</code> field; "
        "it survives only in the optional <code>steps</code> teaching trace.</span><br>"
        f"<span style='color:#60777B;font-size:0.9rem'>Displayed in dependency-first order: {order_text}</span>"
        "</div>"
        "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));"
        "gap:12px;align-items:start'>" + "".join(cards) + "</div></div>"
    ))

Build the **same forward graph**, one readable line per operation. The interactive trace immediately
below is generated from these actual `Value` objects and their stored `ParentLink`s—not from a separate
hand-written event list. On a phone, scroll the graph sideways:

In [6]:
sw = Value(2.0, label="w")
sx = Value(3.0, label="x")
sb = Value(1.0, label="b")
sy = Value(10.0, label="y")

sm = multiply(sw, sx, "m")
sa = add(sm, sb, "a")
se = subtract(sa, sy, "e")
sL = square(se, "L")

print("What m=wx stored during forward:")
for link in sm.parents:
    print(f"  parent {link.value.label}: local ∂m/∂{link.value.label} = {link.local_grad:g}")

safe_order = dependency_safe_order(sL)
print("\nDependency-first order:", " → ".join(node.label for node in safe_order))
print("Backward will process:   ", " → ".join(node.label for node in reversed(safe_order)))

display(draw_graph(sL, show_grad=False))

topology_events = trace_dependency_safe_order(sL)
show_topological_sort_animation(sL, topology_events)

What m=wx stored during forward:
  parent w: local ∂m/∂w = 3
  parent x: local ∂m/∂x = 2

Dependency-first order: w → x → m → b → a → y → e → L
Backward will process:    L → e → y → a → b → m → x → w


Now run backward once, then inspect the result at three levels:

1. the **edge-by-edge trace** shows every chain-rule multiplication and accumulation;
2. the **compact graph** shows the whole computation without overcrowding it;
3. the **complete state cards** expose every field on every `Value`, including every saved parent link.

Only `steps = backward(sL)` performs differentiation. The two `show_...` helpers and `draw_graph` are
teaching displays; removing them would not change any gradient.

In [7]:
steps = backward(sL)
show_backward_steps(steps)

display(draw_graph(sL, show_grad=True))
show_complete_state(sL)

The trace contains every reverse edge. For example, the square sends $-6$ into `e.grad`. On the next
operation, that same stored number becomes the upstream gradient $g_e$ for subtraction.

A single row's product is one **edge contribution to `parent.grad`**—the quantity colored orange in our
legend. If several paths return to one value, each row adds into the same buffer; only their sum is the
full gradient at that parent.

Finally, check that our tiny engine and PyTorch agree at every named value.

In [8]:
scratch_nodes = {"w": sw, "x": sx, "m": sm, "b": sb,
                 "a": sa, "y": sy, "e": se, "L": sL}

for name, node in scratch_nodes.items():
    torch_value, torch_grad = torch_reference[name]
    assert node.data == torch_value
    assert node.grad == torch_grad

print("✓ Every value and gradient matches PyTorch.")

✓ Every value and gradient matches PyTorch.


## 3 · One neuron: fused sigmoid or atomic sigmoid?

Now use a slightly larger graph:

$$
m=wx,\qquad z=m+b,\qquad s=\sigma(z),\qquad e=s-y,\qquad L=e^2.
$$

Choose $w=0.5$, $x=2$, $b=-1$, and $y=1$. Then $z=0$, $s=0.5$, and $L=0.25$,
so the backward numbers stay readable.

We will build the sigmoid in two ways:

- **fused autograd primitive:** one operation $s=\sigma(z)$;
- **atomic graph:** $n=-z$, $q=\exp(n)$, $d=1+q$, and $s=1/d$.

“Fused” here describes the autograd graph: several local steps are packaged behind one operation node.
It does not mean that we are skipping the chain rule.

In [9]:
import math

def negate(u, label):
    return Value(-u.data, label,
                 parents=(ParentLink(u, -1.0),), op="−")

def exponential(u, label):
    out = math.exp(u.data)
    return Value(out, label,
                 parents=(ParentLink(u, out),), op="exp")

def plus_one(u, label):
    return Value(1.0 + u.data, label,
                 parents=(ParentLink(u, 1.0),), op="+1")

def reciprocal(u, label):
    return Value(1.0 / u.data, label,
                 parents=(ParentLink(u, -1.0 / u.data**2),), op="1/x")

def sigmoid(u, label):
    # Stable forward formula; backward reuses the saved output s.
    if u.data >= 0:
        s = 1.0 / (1.0 + math.exp(-u.data))
    else:
        exp_z = math.exp(u.data)
        s = exp_z / (1.0 + exp_z)
    return Value(s, label,
                 parents=(ParentLink(u, s * (1.0 - s)),), op="σ")

The fused rule stores one local derivative:

$$
\frac{\partial s}{\partial z}=s(1-s).
$$

The atomic graph stores four local derivatives. Their product is the same quantity:

$$
\underbrace{\left(-\frac{1}{d^2}\right)}_{s=1/d}
\underbrace{(1)}_{d=1+q}
\underbrace{(q)}_{q=\exp(n)}
\underbrace{(-1)}_{n=-z}
=\frac{q}{d^2}=s(1-s).
$$

In [10]:
def build_sigmoid_neuron(*, fused):
    nodes = {
        "w": Value(0.5, label="w"),
        "x": Value(2.0, label="x"),
        "b": Value(-1.0, label="b"),
        "y": Value(1.0, label="y"),
    }
    nodes["m"] = multiply(nodes["w"], nodes["x"], "m")
    nodes["z"] = add(nodes["m"], nodes["b"], "z")

    if fused:
        nodes["s"] = sigmoid(nodes["z"], "s")
    else:
        nodes["n"] = negate(nodes["z"], "n")
        nodes["q"] = exponential(nodes["n"], "q")
        nodes["d"] = plus_one(nodes["q"], "d")
        nodes["s"] = reciprocal(nodes["d"], "s")

    nodes["e"] = subtract(nodes["s"], nodes["y"], "e")
    nodes["L"] = square(nodes["e"], "L")
    return nodes

fused_nodes = build_sigmoid_neuron(fused=True)
atomic_nodes = build_sigmoid_neuron(fused=False)
fused_steps = backward(fused_nodes["L"])
atomic_steps = backward(atomic_nodes["L"])

display(HTML("<h4>Fused sigmoid · 8 reverse edges</h4>"))
display(draw_graph(fused_nodes["L"], show_grad=True, min_width=1180))
show_backward_steps(fused_steps, highlight_outputs={"s"})

display(HTML("<h4 style='margin-top:24px'>Atomic sigmoid · 11 reverse edges</h4>"))
display(draw_graph(atomic_nodes["L"], show_grad=True, min_width=1700))
show_backward_steps(atomic_steps, highlight_outputs={"s", "d", "q", "n"})

The blue-highlighted cards are the only part that changed:

- fused sigmoid: one update, $g_z=g_s\,s(1-s)=(-1)(0.25)=-0.25$;
- atomic sigmoid: four updates, ending with the same $g_z=-0.25$.

Fusion therefore gives a smaller graph and fewer intermediate gradient buffers. A real library can also
use a numerically stable sigmoid implementation. The mathematics is unchanged: the single fused local
derivative is exactly the product of the four atomic local derivatives.

In [11]:
common = ("w", "x", "m", "b", "z", "s", "y", "e", "L")
for name in common:
    assert math.isclose(fused_nodes[name].data, atomic_nodes[name].data)
    assert math.isclose(fused_nodes[name].grad, atomic_nodes[name].grad)

assert [(s["output"], s["parent"]) for s in fused_steps] == [
    ("L", "e"), ("e", "s"), ("e", "y"), ("s", "z"),
    ("z", "m"), ("z", "b"), ("m", "w"), ("m", "x"),
]
assert [(s["output"], s["parent"]) for s in atomic_steps] == [
    ("L", "e"), ("e", "s"), ("e", "y"), ("s", "d"),
    ("d", "q"), ("q", "n"), ("n", "z"), ("z", "m"),
    ("z", "b"), ("m", "w"), ("m", "x"),
]

expected_sigmoid = {
    "w": (0.5, -0.5), "x": (2.0, -0.125), "m": (1.0, -0.25),
    "b": (-1.0, -0.25), "z": (0.0, -0.25), "s": (0.5, -1.0),
    "y": (1.0, 1.0), "e": (-0.5, -1.0), "L": (0.25, 1.0),
}
for name, (value, grad) in expected_sigmoid.items():
    assert math.isclose(fused_nodes[name].data, value)
    assert math.isclose(fused_nodes[name].grad, grad)

fused_local = next(
    step["local"] for step in fused_steps
    if step["output"] == "s" and step["parent"] == "z"
)
atomic_locals = [
    step["local"] for step in atomic_steps
    if step["output"] in {"s", "d", "q", "n"}
]
assert math.isclose(math.prod(atomic_locals), fused_local)

tw = torch.tensor(0.5, requires_grad=True)
tx = torch.tensor(2.0, requires_grad=True)
tb = torch.tensor(-1.0, requires_grad=True)
ty = torch.tensor(1.0, requires_grad=True)
tL = (torch.sigmoid(tw * tx + tb) - ty) ** 2
tL.backward()

assert math.isclose(fused_nodes["w"].grad, tw.grad.item())
assert math.isclose(fused_nodes["x"].grad, tx.grad.item())
assert math.isclose(fused_nodes["b"].grad, tb.grad.item())
assert math.isclose(fused_nodes["y"].grad, ty.grad.item())

print("✓ Fused, atomic, and PyTorch agree.")
print("  sigmoid local: 4 atomic factors = 1 fused factor =", fused_local)
print("  final gradients: w = -0.5, x = -0.125, b = -0.25, y = 1")

✓ Fused, atomic, and PyTorch agree.
  sigmoid local: 4 atomic factors = 1 fused factor = 0.25
  final gradients: w = -0.5, x = -0.125, b = -0.25, y = 1


## Takeaway

Both systems do the same three things:

1. run the forward operations and store parent links plus local derivatives,
2. start with $g_L=1$ in the loss's `.grad` buffer,
3. compute <span style="color:#2C7A7B;font-weight:700">upstream</span>
   $\times$ <span style="color:#2B6CB0;font-weight:700">local</span>
   $=$ <span style="color:#EB811B;font-weight:700">edge contribution to the parent</span>, then add it to
   the parent's `.grad` buffer.

Our tiny `Value` record and local rules make those steps visible. Fusion does not change the calculus;
it packages a product of local derivatives behind one operation. PyTorch generalizes these ideas to
tensors, neural-network layers, accelerators, and large models.